<a href="https://colab.research.google.com/github/myazzeh/NLP-Course/blob/main/NLP_Transformer_FineTuning_CLF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, DataCollatorWithPadding, TFAutoModelForSequenceClassification
from datasets import load_dataset, Dataset, DatasetDict
import numpy as np
import pandas as pd

#**Download and read your Datasets**##

In [3]:
!wget https://raw.githubusercontent.com/myazzeh/NLP-Course/main/datasets/fake_news_train.csv
!wget https://raw.githubusercontent.com/myazzeh/NLP-Course/main/datasets/fake_news_test.csv

--2025-08-12 07:57:50--  https://raw.githubusercontent.com/myazzeh/NLP-Course/main/datasets/fake_news_train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 296552 (290K) [text/plain]
Saving to: ‘fake_news_train.csv’

fake_news_train.csv 100%[===================>] 289.60K  --.-KB/s    in 0.03s   

2025-08-12 07:57:50 (11.2 MB/s) - ‘fake_news_train.csv’ saved [296552/296552]

--2025-08-12 07:57:51--  https://raw.githubusercontent.com/myazzeh/NLP-Course/main/datasets/fake_news_test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

In [4]:
data = load_dataset("csv",
                    data_files={"train": "/content/fake_news_train.csv",
                                "test":"/content/fake_news_test.csv"})
data

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['claim_s', 'fake_flag'],
        num_rows: 3185
    })
    test: Dataset({
        features: ['claim_s', 'fake_flag'],
        num_rows: 456
    })
})

#**Tokenize training and testing dataset**#

In [5]:
#checkpoint = "CAMeL-Lab/bert-base-arabic-camelbert-mix-sentiment"
checkpoint= 'CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["claim_s"], truncation=True)

tokenized_datasets = data.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/3185 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/456 [00:00<?, ? examples/s]

In [6]:
#tokenizer.decode(1)
tokenizer.convert_tokens_to_ids(['i', 'want', 'to', 'meet', 'you'])

[77, 1, 10220, 1, 1]

In [17]:
tokenizer(["هل ممكن مشاهدة الاحتفالات في مدينة عمان", "الجو مرهق اليوم"], padding=True)

{'input_ids': [[2, 2827, 4050, 4802, 25950, 1912, 3255, 5224, 3], [2, 5697, 3502, 1011, 2217, 3, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 0]]}

#**Convert Datasets to Tensorflow datasets**#

In [8]:
#Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

In [9]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['claim_s', 'fake_flag', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3185
    })
    test: Dataset({
        features: ['claim_s', 'fake_flag', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 456
    })
})

In [10]:
tf_train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols=["fake_flag"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)

tf_validation_dataset = tokenized_datasets["test"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols=["fake_flag"],
    shuffle=False,
    collate_fn=data_collator,
    batch_size=8,
)

/usr/local/lib/python3.11/dist-packages/datasets/arrow_dataset.py:403: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


#**FineTune the transformer, re-train it**#

In [11]:
from keras.losses import SparseCategoricalCrossentropy
from transformers import TFAutoModelForSequenceClassification
model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint)
model.compile(
    optimizer="adam",
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.fit(
    tf_train_dataset,
    validation_data=tf_validation_dataset,
    epochs=2
)

tf_model.h5:   0%|          | 0.00/437M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment were not used when initializing TFBertForSequenceClassification: ['dropout_113']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment.
If your task is sim

Epoch 1/2
399/399 [==============================] - 141s 170ms/step - loss: 0.6964 - accuracy: 0.6298 - val_loss: 0.6515 - val_accuracy: 0.6711
Epoch 2/2
399/399 [==============================] - 44s 111ms/step - loss: 0.6880 - accuracy: 0.6377 - val_loss: 0.6525 - val_accuracy: 0.6711


In [12]:
preds = model.predict(tf_validation_dataset)["logits"]

57/57 [==============================] - 4s 31ms/step


In [13]:
preds

array([[ 2.4208577,  1.2819254, -7.004475 ],
       [ 2.420858 ,  1.2819253, -7.0044756],
       [ 2.4208577,  1.2819253, -7.0044756],
       ...,
       [ 2.4208577,  1.2819253, -7.004475 ],
       [ 2.420858 ,  1.2819253, -7.0044756],
       [ 2.4208577,  1.2819254, -7.0044756]], dtype=float32)

In [14]:
class_preds = np.argmax(preds, axis=1)
print(preds.shape, class_preds.shape)

(456, 3) (456,)


In [15]:
class_preds

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,